In [68]:
import os
import torch
import torch.nn as nn
from collections import defaultdict
from torch.utils.data import Dataset, Subset, DataLoader, ConcatDataset
from CompressedFreezeDataset import CompressedFreezeDataset
from MambaFRZ import initialize_mamba2_predictor
from SmartFRZ import initialize_smartfrz_predictor
import random
import matplotlib.pyplot as plt
from tqdm import tqdm

In [69]:
name_of_experiment = "mambafrz_vgg11_data_generation_12_seeds/training_data_more_data_times_three"
context_window_size = 30
root_dir = f"{name_of_experiment}/context_window_{context_window_size}"
percentage_of_seeds_for_validation = 0.2
frz_predictor_type = "smartfrz"
frz_predictor_path = "mambafrz_vgg11_data_generation_12_seeds/training_data_more_data_times_three/context_window_30/smartfrz_from_scratch_do_not_interfere/smartfrz_9.pth"
re_size = 1024
batch_size = 8

In [70]:
name_of_experiment_2 = "mambafrz_vgg16_data_generation/training_data"
root_dir_2 = f"{name_of_experiment_2}/context_window_{context_window_size}"

In [71]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if frz_predictor_type == "smartfrz":
    in_channel = re_size
    hid_channel = 256
    out_channel = 64
    predictor = initialize_smartfrz_predictor(in_channel, hid_channel, out_channel)
    predictor.load_state_dict(torch.load(frz_predictor_path, map_location=device, weights_only=True))
    predictor.to(device)
elif frz_predictor_type == "mambafrz":
    feature_dim = re_size
    mlp_hid_channel = 512
    mlp_out_channel = 2
    ssm_state_expansion_factor = 32
    projected_dim = feature_dim // 2
    predictor = initialize_mamba2_predictor(feature_dim=feature_dim, projected_dim=projected_dim, ssm_state_expansion_factor=ssm_state_expansion_factor, mlp_hid_channel=mlp_hid_channel, mlp_out_channel=mlp_out_channel)
    predictor.load_state_dict(torch.load(frz_predictor_path, map_location=device, weights_only=True))
    predictor.to(device)

In [76]:
train_dataset = CompressedFreezeDataset(f"{root_dir}/compressed_dataset_{frz_predictor_type}.pkl", frz_predictor_type)
train_dataset_2 = CompressedFreezeDataset(f"{root_dir_2}/compressed_dataset_{frz_predictor_type}.pkl", frz_predictor_type)
train_dataset = ConcatDataset([train_dataset, train_dataset_2])

In [77]:
len(train_dataset)

106560

In [78]:
all_indices = list(range(len(train_dataset)))

In [79]:
seed_to_indices = defaultdict(list)
for idx in range(len(train_dataset)):
    _, _, _, seed = train_dataset[idx][0]
    seed_to_indices[seed].append(idx)
seed_list = list(seed_to_indices.keys())
print("The seeds in the dataset: ", seed_list, len(seed_list))

The seeds in the dataset:  ['5887', '719', '9204', '3575', '2248', '436', '203', '3959', '9296', '1176', '2562', '6256', '7908', '4601', '2093', '2765', '7147', '562', '4273', '1956', '6810', '581', '2200', '3631', '5245', '6921', '5842', '3307', '7830', '2544', '3176', '5325', '5835', '1244', '8388', '7809', '8210', '866', '9151', '1766', '155', '7076', '6140', '5661', '7161', '3554', '6344', '4180', '5289', '1629', '4498', '7778', '5422', '6852', '9855', '2539', '4815', '1648', '2156', '8340', '6732', '1446'] 62


In [80]:
random.seed(1234)
number_of_seeds_for_testing_dataset = int(percentage_of_seeds_for_validation * len(seed_list))
chosen_seeds = set(random.sample(seed_list, number_of_seeds_for_testing_dataset))
val_indices = []
train_indices = []
for seed in chosen_seeds:
    val_indices.extend(seed_to_indices[seed])
training_dataset_seeds = set(seed_list) - chosen_seeds
for seed in training_dataset_seeds:
    train_indices.extend(seed_to_indices[seed])

In [81]:
train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(train_dataset, val_indices)

In [ ]:
seed_prediction_tracker = {} # Each key is a seed to another dict, which stores layers and predictions there
layer_by_layer_accuracy = {} # Get accuracy per layer name, that way it shows overall trends across multiple seeds
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=0)
progress_bar = tqdm(val_loader, desc="Testing Multiple Seed Validation Dataset", leave=False)
val_running_loss = 0.0
val_total_correct = 0
val_total_num = 0
criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)
with torch.no_grad():
    for inputs, labels in progress_bar:
        seed_list = inputs[3]
        layer_names_list = inputs[1]
        epoch_numbers_list = inputs[2]
        inputs, labels = inputs[0].to(device), labels.to(device)
        outputs = predictor(inputs)
        loss = criterion(outputs, labels)
        val_running_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        
        for seed, pred, label, layer_name, epoch_num in zip(seed_list, preds, labels, layer_names_list, epoch_numbers_list):
            if seed not in seed_prediction_tracker:
                seed_prediction_tracker[seed] = {}
            if layer_name not in seed_prediction_tracker[seed]:
                seed_prediction_tracker[seed][layer_name] = []
            # Set layer by layer accuracy defaults
            if layer_name not in layer_by_layer_accuracy:
                layer_by_layer_accuracy[layer_name] = {"total_correct": 0, "total_samples": 0}
            if pred.item() == label.item():
                val_total_correct += 1
                layer_by_layer_accuracy[layer_name]["total_correct"] += 1
            seed_prediction_tracker[seed][layer_name].append((epoch_num, pred.item(), label.item()))
            val_total_num += 1
            layer_by_layer_accuracy[layer_name]["total_samples"] += 1
    for seed in seed_prediction_tracker.keys():
        for layer_name, frz_predictions_by_layer in seed_prediction_tracker[seed].items():
            frz_predictions_by_layer.sort(key=lambda item: int(item[0]))
            epoch_list = [int(item[0]) for item in frz_predictions_by_layer]
            frz_predictor_list = [item[1] for item in frz_predictions_by_layer]
            label_predictor_list = [item[2] for item in frz_predictions_by_layer]
            
            plt.title(f"Seed {seed}, Layer {layer_name} Predictions")
            name_of_predictor = "MambaFRZ" if frz_predictor_type == "mambafrz" else "SmartFRZ"
            plt.plot(epoch_list, frz_predictor_list, label=f"{name_of_predictor} Predictions")
            plt.plot(epoch_list, label_predictor_list, label="Labels")
            plt.legend()
            plt.show()
    
    for layer_name, layer_accuracy in layer_by_layer_accuracy.items():
        acc = layer_accuracy["total_correct"] / layer_accuracy["total_samples"]
        print(f"{layer_name}: {acc:.4f}, {layer_accuracy['total_correct']} / {layer_accuracy['total_samples']}")
    print(f"Overall Validation Accuracy: {(val_total_correct / val_total_num):.4f}")